In [7]:
import pandas as pd
from pathlib import Path

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"
OUTPUT_CSV = Path("data") / "government_response.csv"

# =============================================================================
# LOAD DATA
# =============================================================================

df = pd.read_csv(INPUT_CSV)

print("Input shape:", df.shape)

# =============================================================================
# PARAMETERS
# =============================================================================

FISCAL_YEAR_START_MONTH = 4  # April

# =============================================================================
# Z-SCORE FUNCTION
# =============================================================================

def zscore(x):
    std = x.std(ddof=0)

    if std == 0:
        return pd.Series(0, index=x.index)

    return (x - x.mean()) / std


# =============================================================================
# INVERTED SD-INTERVAL CLASSIFICATION
# =============================================================================

def classify(z):
    """
    Government Response:
    Higher cumulative tender spending = stronger response capacity
    Therefore invert the classes:
        1 = strongest response
        5 = weakest response
    """

    if z <= -1.5:
        return 5
    elif z <= -0.5:
        return 4
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 2
    else:
        return 1


# =============================================================================
# FINANCIAL YEAR FUNCTION
# =============================================================================

def get_financial_year(timeperiod):

    tp = str(timeperiod).replace("_", "-")

    year, month = tp.split("-")

    year = int(year)
    month = int(month)

    if month >= FISCAL_YEAR_START_MONTH:
        return f"{year}-{year + 1}"
    else:
        return f"{year - 1}-{year}"


# =============================================================================
# STEP 1
# UPSCALE BLOCK TENDERS -> DISTRICT-MONTH TENDERS
# =============================================================================

district_df = (
    df.groupby(
        ["district", "timeperiod"],
        as_index=False
    )
    .agg(
        district_tender_value=(
            "total_tender_awarded_value",
            "sum"
        )
    )
)

print("\nDistrict-month rows:", len(district_df))

# =============================================================================
# STEP 2
# FINANCIAL YEAR
# =============================================================================

district_df["financial_year"] = (
    district_df["timeperiod"]
    .apply(get_financial_year)
)

# =============================================================================
# STEP 3
# SORT CHRONOLOGICALLY
# =============================================================================

district_df["date"] = pd.to_datetime(
    district_df["timeperiod"],
    format="%Y_%m"
)

district_df = district_df.sort_values(
    ["district", "date"]
)

# =============================================================================
# STEP 4
# CUMULATIVE DISTRICT TENDER VALUE WITHIN FINANCIAL YEAR
# =============================================================================

district_df["cum_tender_value"] = (
    district_df.groupby(
        ["district", "financial_year"]
    )["district_tender_value"]
    .cumsum()
)

# =============================================================================
# STEP 5
# MONTH-WISE Z-SCORES ACROSS DISTRICTS
# =============================================================================

district_df["govtresponse_z"] = (
    district_df.groupby("timeperiod")["cum_tender_value"]
    .transform(zscore)
)

# =============================================================================
# STEP 6
# GOVERNMENT RESPONSE CLASS
# =============================================================================

district_df["government_response"] = (
    district_df["govtresponse_z"]
    .apply(classify)
)

# =============================================================================
# STEP 7
# EXPAND BACK TO OBJECT_ID LEVEL
# =============================================================================

government_response_df = (
    df[
        [
            "object_id",
            "district",
            "timeperiod",
        ]
    ]
    .drop_duplicates()
    .merge(
        district_df[
            [
                "district",
                "timeperiod",
                "district_tender_value",
                "cum_tender_value",
                "govtresponse_z",
                "government_response",
            ]
        ],
        on=["district", "timeperiod"],
        how="left",
    )
)

# =============================================================================
# STEP 8
# SAVE GOVERNMENT_RESPONSE.CSV
# =============================================================================

government_response_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nSaved:", OUTPUT_CSV)

# =============================================================================
# STEP 9
# APPEND TO MASTER_VARIABLES.CSV USING OBJECT_ID + TIMEPERIOD
# =============================================================================

master = df.copy()

master = master.merge(
    government_response_df[
        [
            "object_id",
            "timeperiod",
            "government_response",
        ]
    ],
    on=["object_id", "timeperiod"],
    how="left",
)

master.to_csv(
    INPUT_CSV,
    index=False
)

print("\nUpdated:", INPUT_CSV)

# =============================================================================
# CHECKS
# =============================================================================

print("\nGovernment Response Distribution")
print(
    government_response_df["government_response"]
    .value_counts()
    .sort_index()
)

print("\nMissing scores:")
print(
    government_response_df["government_response"]
    .isna()
    .sum()
)

print("\nPreview:")
print(
    government_response_df.head()
)

Input shape: (7222, 27)

District-month rows: 690

Saved: data/government_response.csv

Updated: data/MASTER_VARIABLES.csv

Government Response Distribution
government_response
1     385
2     218
3    5657
4     962
Name: count, dtype: int64

Missing scores:
0

Preview:
      object_id district timeperiod  district_tender_value  cum_tender_value  \
0  21-384-03276   Anugul    2023_01                    0.0               0.0   
1  21-384-03277   Anugul    2023_01                    0.0               0.0   
2  21-384-03278   Anugul    2023_01                    0.0               0.0   
3  21-384-03279   Anugul    2023_01                    0.0               0.0   
4  21-384-03280   Anugul    2023_01                    0.0               0.0   

   govtresponse_z  government_response  
0       -0.253089                    3  
1       -0.253089                    3  
2       -0.253089                    3  
3       -0.253089                    3  
4       -0.253089                    3  
